# Vacinas em Destaque

Análise exploratória da distribuição dos imunizantes na base de vacinação.

## 1. Bibliotecas e configuração

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

## 2. Carregamento da base

A base está no caminho definido pelo projeto.

In [ ]:
file_path = "../../data/processed/vacinas_analitico.csv"
df = pd.read_csv(file_path)
df.head()

## 3. Visão geral da base

In [ ]:
print(f"Registros: {df.shape[0]:,}".replace(",", "."))
print(f"Colunas: {df.shape[1]}")

display(df.head())

## 4. Qualidade dos dados

Verificamos valores ausentes e registros duplicados.

In [ ]:
print("Valores ausentes por coluna:")
display(df.isna().sum().sort_values(ascending=False).head(15))

print("\nRegistros duplicados:", df.duplicated().sum())

## 5. Tratamento da data

In [ ]:
df["data_da_aplicacao"] = pd.to_datetime(
    df["data_da_aplicacao"], dayfirst=True, errors="coerce"
)
df["ano"] = df["data_da_aplicacao"].dt.year
df["mes"] = df["data_da_aplicacao"].dt.to_period("M").astype(str)

print("Período:", df["data_da_aplicacao"].min(), "até", df["data_da_aplicacao"].max())

## 6. Distribuição dos imunizantes

Quais vacinas aparecem com maior frequência na base?

In [ ]:
vacinas = (
    df["nome_da_vacina"]
    .value_counts()
    .rename_axis("Vacina")
    .reset_index(name="Aplicações")
)

vacinas["Percentual (%)"] = vacinas["Aplicações"] / len(df) * 100
display(vacinas.head(10))

## 7. Ranking das vacinas

In [ ]:
top_vacinas = vacinas.head(10).sort_values("Aplicações")

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(
    top_vacinas["Vacina"],
    top_vacinas["Aplicações"],
    color="#4472C4"
)

ax.set_title("Vacinas mais aplicadas", fontsize=17, fontweight="bold", pad=15)
ax.set_xlabel("Número de aplicações")
ax.set_ylabel("")

for bar, valor in zip(bars, top_vacinas["Aplicações"]):
    ax.text(
        bar.get_width() + top_vacinas["Aplicações"].max() * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{valor:,}".replace(",", "."),
        va="center",
        fontsize=10,
        fontweight="bold"
    )

ax.spines[["top", "right", "left"]].set_visible(False)
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

## 8. Participação percentual dos imunizantes

## Concentração das aplicações

A curva de concentração mostra quanto dos registros está concentrado nos imunizantes mais frequentes.

In [ ]:
pareto = vacinas.copy()
pareto["Acumulado (%)"] = pareto["Percentual (%)"].cumsum()

fig, ax1 = plt.subplots(figsize=(12, 6))

x = range(len(pareto))
ax1.bar(
    x,
    pareto["Percentual (%)"],
    color="#4472C4",
    alpha=0.85
)

ax1.set_xlabel("Imunizantes em ordem de frequência")
ax1.set_ylabel("Participação individual (%)")
ax1.set_title(
    "Concentração das aplicações",
    fontsize=17,
    fontweight="bold",
    pad=15
)

# Linha acumulada em eixo secundário
ax2 = ax1.twinx()
ax2.plot(
    x,
    pareto["Acumulado (%)"],
    color="#ED7D31",
    marker="o",
    linewidth=2.5,
    markersize=4
)
ax2.set_ylabel("Participação acumulada (%)")
ax2.set_ylim(0, 105)

# Mostra poucos rótulos no eixo X para evitar poluição visual
n = len(pareto)
passo = max(1, n // 10)
indices = list(range(0, n, passo))

ax1.set_xticks(indices)
ax1.set_xticklabels(
    [str(pareto.iloc[i]["Vacina"])[:18] for i in indices],
    rotation=45,
    ha="right"
)

ax1.grid(axis="y", alpha=0.2)
ax1.spines["top"].set_visible(False)
ax2.spines["top"].set_visible(False)

plt.tight_layout()
plt.show()

print(
    f"Os 5 imunizantes mais frequentes representam "
    f"{pareto.head(5)['Percentual (%)'].sum():.1f}% dos registros."
)

In [ ]:
top = vacinas.head(10).sort_values("Percentual (%)")

fig, ax = plt.subplots(figsize=(11, 6))
bars = ax.barh(
    top["Vacina"],
    top["Percentual (%)"],
    color="#70AD47"
)

ax.set_title("Participação das vacinas", fontsize=17, fontweight="bold", pad=15)
ax.set_xlabel("Participação (%)")
ax.set_ylabel("")

for bar, valor in zip(bars, top["Percentual (%)"]):
    ax.text(
        bar.get_width() + 0.15,
        bar.get_y() + bar.get_height() / 2,
        f"{valor:.1f}%",
        va="center",
        fontsize=10,
        fontweight="bold"
    )

ax.spines[["top", "right", "left"]].set_visible(False)
ax.grid(axis="x", alpha=0.25)
plt.tight_layout()
plt.show()

## 9. Distribuição por estratégia

In [ ]:
estrategias = (
    df["estrategia"]
    .value_counts()
    .rename_axis("Estratégia")
    .reset_index(name="Aplicações")
)

display(estrategias)

fig, ax = plt.subplots(figsize=(10, 5.5))
bars = ax.bar(
    estrategias["Estratégia"],
    estrategias["Aplicações"],
    color="#5B9BD5"
)

ax.set_title("Aplicações por estratégia", fontsize=17, fontweight="bold", pad=15)
ax.set_ylabel("Aplicações")
ax.set_xlabel("")
plt.xticks(rotation=30, ha="right")

for bar, valor in zip(bars, estrategias["Aplicações"]):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{valor:,}".replace(",", "."),
        ha="center",
        va="bottom",
        fontsize=9,
        fontweight="bold"
    )

ax.spines[["top", "right"]].set_visible(False)
ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

## 10. Vacina por estratégia

In [ ]:
tabela_vacina_estrategia = pd.crosstab(
    df["nome_da_vacina"], df["estrategia"]
)

display(tabela_vacina_estrategia.head(15))

## 11. Mapa de calor

A visualização ajuda a identificar quais imunizantes aparecem associados a cada estratégia.

In [ ]:
# Prepara os dados para o mapa
heatmap = pd.crosstab(
    df["nome_da_vacina"],
    df["estrategia"]
)

# Mantém apenas as 12 vacinas mais frequentes
top_nomes = vacinas.head(12)["Vacina"].tolist()
heatmap = heatmap.reindex(
    index=[v for v in top_nomes if v in heatmap.index],
    fill_value=0
)

# Remove linhas e colunas sem registros
heatmap = heatmap.loc[
    heatmap.sum(axis=1) > 0,
    heatmap.sum(axis=0) > 0
]

if heatmap.empty:
    print("Não há dados suficientes para gerar o mapa de calor.")
else:
    fig, ax = plt.subplots(figsize=(12, 6.5))

    sns.heatmap(
        heatmap,
        annot=True,
        fmt=".0f",
        cmap="YlOrRd",
        linewidths=0.8,
        linecolor="white",
        cbar_kws={"label": "Aplicações"},
        ax=ax
    )

    ax.set_title("Vacinas por estratégia", fontsize=16, fontweight="bold", pad=15)
    ax.set_xlabel("Estratégia", fontsize=11)
    ax.set_ylabel("Vacina", fontsize=11)

    plt.xticks(rotation=30, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

## 12. Evolução mensal

Observamos como o volume de aplicações dos principais imunizantes varia ao longo do período.

In [ ]:
principais = vacinas.head(5)["Vacina"]
mensal = (
    df[df["nome_da_vacina"].isin(principais)]
    .groupby(["mes", "nome_da_vacina"])
    .size()
    .unstack(fill_value=0)
)

display(mensal.head())

## 13. Gráfico da evolução mensal

In [ ]:
ax = mensal.plot(
    figsize=(12, 6),
    marker="o",
    linewidth=2.5,
    markersize=6
)

ax.set_title("Evolução das aplicações", fontsize=17, fontweight="bold", pad=15)
ax.set_xlabel("Mês")
ax.set_ylabel("Aplicações")
ax.spines[["top", "right"]].set_visible(False)
ax.grid(alpha=0.2)

plt.xticks(rotation=45, ha="right")
plt.legend(
    title="Vacina",
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False
)

plt.tight_layout()
plt.show()

## 14. Distribuição por bairro

In [ ]:
colunas_bairro = [c for c in df.columns if "bairro" in c.lower()]

print("Colunas relacionadas a bairro:", colunas_bairro)

if not colunas_bairro:
    print("A base não possui uma coluna de bairro. A análise por bairro será ignorada.")
else:
    coluna_bairro = colunas_bairro[0]
    bairros = (
        df[coluna_bairro]
        .dropna()
        .value_counts()
        .rename_axis("Bairro")
        .reset_index(name="Aplicações")
    )
    display(bairros.head(10))

## 15. Principais bairros

In [ ]:
if "bairros" in globals() and not bairros.empty:
    top_bairros = bairros.head(10).sort_values("Aplicações")

    fig, ax = plt.subplots(figsize=(11, 6))
    bars = ax.barh(
        top_bairros["Bairro"],
        top_bairros["Aplicações"],
        color="#ED7D31"
    )

    ax.set_title("Bairros com mais aplicações", fontsize=17, fontweight="bold", pad=15)
    ax.set_xlabel("Número de aplicações")
    ax.set_ylabel("")

    for bar, valor in zip(bars, top_bairros["Aplicações"]):
        ax.text(
            bar.get_width() + top_bairros["Aplicações"].max() * 0.01,
            bar.get_y() + bar.get_height() / 2,
            f"{valor:,}".replace(",", "."),
            va="center",
            fontsize=10,
            fontweight="bold"
        )

    ax.spines[["top", "right", "left"]].set_visible(False)
    ax.grid(axis="x", alpha=0.25)
    plt.tight_layout()
    plt.show()
else:
    print("Gráfico não gerado: não foi encontrada uma coluna de bairro.")

## 16. Distribuição por dose

In [ ]:
doses = (
    df["dose"]
    .dropna()
    .astype(str)
    .value_counts()
    .rename_axis("Dose")
    .reset_index(name="Aplicações")
)

display(doses.head(15))

# Mostra apenas as 10 categorias mais frequentes para manter a leitura.
top_doses = doses.head(10).sort_values("Aplicações")

fig, ax = plt.subplots(figsize=(11, 6.5))

bars = ax.barh(
    top_doses["Dose"],
    top_doses["Aplicações"],
    color="#4472C4"
)

ax.set_title("Principais doses", fontsize=17, fontweight="bold", pad=15)
ax.set_xlabel("Número de aplicações")
ax.set_ylabel("")

for bar, valor in zip(bars, top_doses["Aplicações"]):
    ax.text(
        bar.get_width() + top_doses["Aplicações"].max() * 0.01,
        bar.get_y() + bar.get_height() / 2,
        f"{valor:,}".replace(",", "."),
        va="center",
        fontsize=10,
        fontweight="bold"
    )

ax.spines[["top", "right", "left"]].set_visible(False)
ax.grid(axis="x", alpha=0.25)

plt.tight_layout()
plt.show()

## 17. Combinação entre vacina, estratégia e dose

In [ ]:
combinacoes = (
    df.groupby(["nome_da_vacina", "estrategia", "dose"])
    .size()
    .reset_index(name="Aplicações")
    .sort_values("Aplicações", ascending=False)
)

display(combinacoes.head(15))

## 18. Concentração das aplicações

Verificamos quanto das aplicações está concentrado nos principais imunizantes.

In [ ]:
vacinas["Acumulado (%)"] = vacinas["Percentual (%)"].cumsum()

display(vacinas.head(10))

## 19. Síntese dos resultados

In [ ]:
print("Total de aplicações:", len(df))
print("Quantidade de imunizantes:", df["nome_da_vacina"].nunique())
print("Imunizante mais frequente:", vacinas.iloc[0]["Vacina"])
print(
    "Participação do mais frequente:",
    round(vacinas.iloc[0]["Percentual (%)"], 2),
    "%"
)
print("Quantidade de estratégias:", df["estrategia"].nunique())

if "colunas_bairro" in globals() and colunas_bairro:
    print("Quantidade de bairros:", df[colunas_bairro[0]].nunique())
else:
    print("Quantidade de bairros: não disponível na base")

## Síntese dos principais achados

In [ ]:
total_aplicacoes = len(df)
vacina_principal = vacinas.iloc[0]["Vacina"]
participacao_principal = vacinas.iloc[0]["Percentual (%)"]

print(f"Total de aplicações analisadas: {total_aplicacoes:,}".replace(",", "."))
print(f"Imunizante mais frequente: {vacina_principal}")
print(f"Participação do principal imunizante: {participacao_principal:.1f}%")
print(f"Quantidade de imunizantes: {df['nome_da_vacina'].nunique()}")
print(f"Quantidade de estratégias: {df['estrategia'].nunique()}")

if "colunas_bairro" in globals() and colunas_bairro:
    print(f"Quantidade de bairros: {df[colunas_bairro[0]].nunique()}")

## 20. Limitações

- Os registros representam aplicações, não necessariamente pessoas únicas.
- A frequência de aplicações não representa, sozinha, cobertura vacinal.
- Os resultados dependem do período e da qualidade dos dados disponíveis.
- A análise descreve a distribuição dos registros e não mede eficácia clínica dos imunizantes.

## Conclusão

A análise exploratória permite identificar quais imunizantes possuem maior presença na base, como as aplicações se distribuem entre estratégias, doses e bairros e como os registros variam ao longo do tempo.

Os gráficos de ranking, evolução temporal, relação entre vacina e estratégia e concentração das aplicações ajudam a responder às principais questões propostas, mantendo a interpretação baseada exclusivamente nos registros disponíveis.